# 04 · MentalGLM (Zhai et al.)

The released MentalGLM-chat (GLM-4-9B), QLoRA-tuned per MATCHA task with label-only answers
(`configs/llm.yaml`), plus zero-shot rows. Deviations from the paper (4-bit instead of fp16, 2
instead of 10 epochs, label-only answers because of the DUA, per-task adapters):
`docs/BASELINES.md` §2.

| | |
|---|---|
| Hardware | one GPU with 16 GB (T4); fp16 sanity check: ≥ 24 GB or two 16 GB GPUs |
| Disk | ~40 GB (the 18.8 GB checkpoint and its converted copy) |
| Time (measured on a T4) | sanity check ~1 h; per fine-tuning run 11.3 h (post tasks), 8 h (multiclass), 2.5 h (binary): ~100 GPU-h for 12 runs; zero-shot ~2.7 h per model |
| Needs | notebook 01; internet for Hugging Face (`zwzzz/MentalGLM-chat`, `zai-org/glm-4-9b-chat`) |
| Writes | `outputs/llm/{sanity,finetune,zeroshot}/` |

A fine-tuning run saves after every epoch and resumes from there. The same work in one command:
`python scripts/run_pipeline.py --no-limit --config configs/pipeline_mentalglm.yaml`.

> **Data use.** MATCHA is under a Data Use Agreement. These notebooks print counts and scores only, never posts; keep it that way if you add cells, and do not commit notebooks with outputs.

In [ ]:
import sys
from pathlib import Path

# make `matcha_bench` importable from a fresh clone (not needed after `pip install -e .`)
sys.path.insert(0, str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"))
from matcha_bench.notebook import check, setup, step

ROOT = setup()  # every step runs from the repository root
print(ROOT)

In [ ]:
step("-m", "pip", "install", "peft>=0.13", "bitsandbytes>=0.45", "transformers>=4.46")

## 1. Download and convert the checkpoint

The released weights ship code for transformers 4.43, which no longer runs. They are converted to
the native transformers `glm` format (`src/matcha_bench/llm/convert.py`; identical outputs to the
original code on small test models).

In [ ]:
step("scripts/run_llm_eval.py", "--prepare-only")

## 2. Sanity check: MentalGLM-chat on the paper's own test sets

No training: the released model labels SOS-HL-1K (Suicide, binary F1) and SocialCD-3K (Cognitive,
micro-F1). Paper: 85.12 / 71.04. Ours: fp16 82.95 / 69.56; 4-bit (the setting used for MATCHA)
84.13 / 67.25.

In [ ]:
step("scripts/run_llm_eval.py", "--model", "mentalglm-chat", "--quant", "4bit")
# with a >= 24 GB GPU (or two 16 GB GPUs) also the released precision:
# step("scripts/run_llm_eval.py", "--model", "mentalglm-chat", "--quant", "fp16")

In [ ]:
import json

for task, reported in [("suicide", 84.13), ("cognitive", 67.25)]:
    m = json.loads(Path(f"outputs/llm/sanity/mentalglm-chat__4bit/{task}/metrics.json").read_text(encoding="utf-8"))
    check(f"4-bit {task} F1", round(100 * m["f1"], 2), reported, tol=1.0)

## 3. QLoRA fine-tuning on the MATCHA tasks

Batch 4 × 8, lr 1e-4, 1% warm-up, LoRA rank 8, every training example, 2 epochs, best epoch on
the full dev set. `user_binary_masked` uses the tag `mentalglm-chat`, whose caps (4,000 training
examples, 500 dev) exceed that task's 2,392 training and 358 dev users, so it is the same recipe.

**Seeds.** 42, 43, 44, except `disorder_post`: our seed 44 diverged twice (non-finite fp16
activations; the guard stopped the run and wrote `diverged.json`), and by a rule fixed before the
rerun the cell uses the next unused seed, 45.

In [ ]:
step("scripts/run_llm_finetune.py", "--tasks", "matcha_user_binary_masked", "--models", "mentalglm-chat")
step("scripts/run_llm_finetune.py", "--tasks", "matcha_disorder_post_masked", "matcha_user_multiclass_masked",
     "--models", "mentalglm-chat-full")
step("scripts/run_llm_finetune.py", "--tasks", "matcha_disorder_post", "--models", "mentalglm-chat-full",
     "--seeds", "42", "43", "45")

## 4. Zero-shot rows

The same prompt, inputs, 4-bit weights and answer parsing, no training; greedy decoding, so one run
per model. An answer that names no label counts as wrong. Answer cap: 24 tokens for MentalGLM-chat,
96 for GLM-4-9B-chat, whose answers open with a preamble (`configs/llm_zeroshot.yaml`).

In [ ]:
MATCHA_TASKS = ["matcha_disorder_post", "matcha_disorder_post_masked",
                "matcha_user_binary_masked", "matcha_user_multiclass_masked"]
step("scripts/run_llm_zeroshot.py", "--tasks", *MATCHA_TASKS, "--models", "mentalglm-chat", "glm-4-9b-chat")

## 5. Scores

**Reported (test macro-F1, %):**

| Model | disorder_post | disorder_post_masked | user_binary_masked | user_multiclass_masked |
|---|---|---|---|---|
| MentalGLM-chat, fine-tuned | 60.81 ± 3.14 (64.09 / 57.83 / 60.52) | 51.42 ± 1.05 (52.58 / 50.55 / 51.13) | 88.98 ± 0.25 | 54.83 ± 0.83 |
| MentalGLM-chat, zero-shot | 41.58 | 29.74 | 49.52 | 20.74 |
| GLM-4-9B-chat, zero-shot | 31.34 | 17.34 | 19.41 | 6.87 |

In [ ]:
rows = []
for kind, tags in [("finetune", ["mentalglm-chat-full", "mentalglm-chat"]), ("zeroshot", ["mentalglm-chat", "glm-4-9b-chat"])]:
    for path in sorted(Path(f"outputs/llm/{kind}").glob("matcha_*/*/seed*/metrics.json")):
        task, tag, seed = path.parts[-4:-1]
        if tag in tags:
            m = json.loads(path.read_text(encoding="utf-8"))
            rows.append({"kind": kind, "task": task, "model": tag, "seed": seed,
                         "f1": round(100 * m["test"]["f1"], 2), "unparsed": m["test"].get("unparsed")})
import pandas as pd

print(pd.DataFrame(rows).to_string(index=False))

Next: `05_roberta.ipynb`.